## PertAdapt on VS Code + a rented GPU box (RunPod / Lambda / Paperspace)

End-to-end pipeline: GCS data access, PertAdapt setup, preprocessing, and training a scFoundation-backed
adapter on the K562 essential single-cell dataset.

**One-time setup per fresh GPU box:**
1. Connect VS Code to the box via the **Remote-SSH** extension (local Macs have no CUDA GPU).
2. `PertAdapt/` is a git submodule of this repo, not something to `git clone` separately - if it's
   ever empty, run `git submodule update --init --recursive`.
3. The scFoundation checkpoint (`PertAdapt/scFoundation/model/models/models.ckpt`, ~1.4GB) is too
   large for git - copy it in separately (e.g. `scp` from wherever you downloaded it) before running
   the training cells near the end.
4. Run `gcloud auth login && gcloud auth application-default login` once per fresh box for GCS access
   (the cell below installs the CLI itself and checks whether this is done).
5. Rental GPU templates (e.g. RunPod's "PyTorch" image) usually ship a torch+CUDA build that already
   matches the driver - the install cell below checks before reinstalling anything.

### 1. GPU check & repo root

In [1]:
import os
import subprocess

gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("WARNING: nvidia-smi failed - no GPU visible on this machine. Training will not work here.")
    print(gpu_check.stderr)
else:
    print(gpu_check.stdout)

# Every path below is relative to the repo root rather than a hardcoded Colab-style /content
# directory, so this notebook works the same regardless of where the GPU box checked it out.
REPO_ROOT = subprocess.run(
    ['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True, check=True
).stdout.strip()
PERTADAPT_ROOT = f"{REPO_ROOT}/PertAdapt"
DATA_DIR = f"{REPO_ROOT}/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("PERTADAPT_ROOT:", PERTADAPT_ROOT)
print("DATA_DIR:", DATA_DIR)

Wed Sep 30 03:14:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:41:00.0 Off |                    0 |
| N/A   24C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Install the `gcloud` CLI

In [ ]:
import os
import subprocess
import shutil

# Rented GPU boxes (RunPod/Lambda/Paperspace/etc.) don't ship the gcloud CLI the way a GCE VM
# image does, but it's needed below to read the gs://2026-vcc-llsa bucket.
gcloud_sdk_bin = os.path.expanduser('~/google-cloud-sdk/bin')
gcloud_installed_on_disk = os.path.exists(os.path.join(gcloud_sdk_bin, 'gcloud'))

if shutil.which('gcloud') is None and not gcloud_installed_on_disk:
    # Only actually missing (not just missing from this fresh kernel's PATH) - install it.
    print("gcloud CLI not found - installing...")
    subprocess.run(
        'curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts',
        shell=True, check=True,
    )

if shutil.which('gcloud') is None:
    # Either just installed above, or already present on disk from a prior kernel session -
    # either way this fresh kernel process's PATH doesn't know about it yet.
    os.environ['PATH'] = gcloud_sdk_bin + os.pathsep + os.environ['PATH']

subprocess.run(['gcloud', '--version'], check=True)

### 3. PertAdapt code

Already vendored as a git submodule at `PertAdapt/` - no separate clone needed.

In [ ]:
assert os.path.isdir(PERTADAPT_ROOT), (
    f"{PERTADAPT_ROOT} is missing or empty - run `git submodule update --init --recursive` "
    "in a terminal on this box"
)

# modules/encoders.py does `sys.path.append("../model/")` then `from load import *`, relative to
# whatever the current working directory happens to be at import time - it never resolves to
# PertAdapt's own scFoundation/model/load.py on its own. A symlink makes that relative path
# actually land somewhere real, without touching PertAdapt's source.
pertadapter_model_link = f"{PERTADAPT_ROOT}/scFoundation/PertAdapter/model"
scfoundation_model_dir = f"{PERTADAPT_ROOT}/scFoundation/model"
if not os.path.exists(pertadapter_model_link):
    os.symlink(scfoundation_model_dir, pertadapter_model_link)
    print(f"Created symlink: {pertadapter_model_link} -> {scfoundation_model_dir}")
else:
    print(f"Symlink already present: {pertadapter_model_link}")

### 4. Install Python dependencies

In [ ]:
import sys

!{sys.executable} -m pip install -q anndata scanpy dcor local-attention

In [ ]:
CUDA_TAG = "cu121"  # <-- EDIT THIS after checking nvidia-smi on your GPU box

# Using `sys.executable -m pip` (not a bare `pip`/`!pip`) guarantees this installs into whatever
# Python this notebook's kernel is actually running, even if a different Python/pip is on PATH.
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/{CUDA_TAG}

# Core data science libraries. anndata/scikit-learn kept at environment.yml's pinned versions.
!{sys.executable} -m pip install numpy pandas scipy anndata==0.12.6 scikit-learn==1.5.0
!{sys.executable} -m pip install einops==0.8.2 matplotlib==3.9.1

# Deferred: transformers/datasets aren't used by prep_k562_bulk.py; install later if PertAdapt's
# training scripts turn out to need them.
# !{sys.executable} -m pip install transformers==4.43.3 datasets==2.20.0

In [ ]:
import torch
# Detect the ACTUAL installed torch + CUDA version rather than hardcoding one - a hardcoded string
# breaks the moment torch's version differs from whatever machine this was last run on.
TORCH = torch.__version__.split('+')[0]                                  # e.g. "2.11.0"
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'  # e.g. "cu128"
pyg_url = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"
print(f"Detected torch {TORCH} / {CUDA} -> using PyG wheel index: {pyg_url}")

!{sys.executable} -m pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_url}
!{sys.executable} -m pip install torch_geometric

import torch_geometric
print("torch_geometric installed:", torch_geometric.__version__)

In [ ]:
# PertAdapt's gears/ vendor code imports DataLoader from the pre-2.x torch_geometric location;
# newer torch_geometric moved it to .loader.
!grep -rl "from torch_geometric.data import DataLoader" {PERTADAPT_ROOT}/scFoundation/PertAdapter/gears/ \
  | xargs sed -i 's/from torch_geometric.data import DataLoader/from torch_geometric.loader import DataLoader/'

### 5. Authenticate to Google Cloud

In [ ]:
# Colab's browser popup (google.colab.auth) doesn't exist outside Colab, and unlike a GCE VM, a
# rented GPU box has no built-in GCP service account either - so a fresh box always needs one
# explicit login. This checks whether that's already been done, and falls back to an interactive
# `gcloud auth login` if not (works fine over an SSH terminal in VS Code).
result = subprocess.run(['gcloud', 'auth', 'list', '--format=value(account)'], capture_output=True, text=True)
active_accounts = result.stdout.strip()
if active_accounts:
    print(f"Authenticated as: {active_accounts}")
else:
    print("No active gcloud credentials found - run this in the VS Code terminal (not this cell):")
    print("  gcloud auth login && gcloud auth application-default login")

In [ ]:
!gcloud config set project virtual-cell-challenge-2026

### 6. Browse the training data bucket

In [2]:
BUCKETtrain = "gs://2026-vcc-llsa/train/"
!gcloud storage ls {BUCKETtrain}*.h5ad


gs://2026-vcc-llsa/train/GSE264667_hepg2_raw_singlecell_01.h5ad
gs://2026-vcc-llsa/train/GSE264667_jurkat_raw_singlecell_01.h5ad
gs://2026-vcc-llsa/train/K562_essential_normalized_singlecell_01.h5ad
gs://2026-vcc-llsa/train/K562_essential_raw_bulk_01.h5ad
gs://2026-vcc-llsa/train/K562_essential_raw_singlecell_01.h5ad
gs://2026-vcc-llsa/train/K562_gwps_normalized_bulk_01.h5ad
gs://2026-vcc-llsa/train/K562_gwps_normalized_singlecell_01.h5ad
gs://2026-vcc-llsa/train/K562_gwps_raw_bulk_01.h5ad
gs://2026-vcc-llsa/train/K562_gwps_raw_singlecell_01.h5ad
gs://2026-vcc-llsa/train/adata_Training.h5ad
gs://2026-vcc-llsa/train/rpe1_normalized_bulk_01.h5ad
gs://2026-vcc-llsa/train/rpe1_normalized_singlecell_01.h5ad
gs://2026-vcc-llsa/train/rpe1_raw_bulk_01.h5ad


### 7. Download and peek at one training file

In [15]:
import anndata as ad

specific_file_path = "gs://2026-vcc-llsa/train/K562_essential_raw_singlecell_01.h5ad"
filename = os.path.basename(specific_file_path)
local_path = f"{DATA_DIR}/{filename}"

if not os.path.exists(local_path):
    print(f"Downloading {specific_file_path} -> {local_path}")
    !gcloud storage cp {specific_file_path} {local_path}
else:
    print(f"Already downloaded: {local_path}")

adata_specific = ad.read_h5ad(local_path, backed='r')
print(f"Dimensions: {adata_specific.shape}")
print("obs columns:", adata_specific.obs.columns.tolist())
print("var columns:", adata_specific.var.columns.tolist())

# backed='r' keeps an open h5py handle on local_path for as long as this kernel lives - on Linux,
# a deleted-but-still-open file keeps its disk space allocated, so an os.remove(local_path) later
# (in the preprocessing cell) silently fails to actually free anything until this closes.
adata_specific.file.close()

Copying gs://2026-vcc-llsa/train/K562_essential_raw_singlecell_01.h5ad to file:///root/vcc2026-l3ba/data/K562_essential_raw_singlecell_01.h5ad
  Completed files 1/1 | 9.9GiB/9.9GiB | 1.6GiB/s                               

Average throughput: 1.4GiB/s
Dimensions: (310385, 8563)
obs columns: ['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count']
var columns: ['gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano']


### 8. Preprocess the K562 essential single-cell file into GEARS' schema

In [16]:
import scanpy as sc

# train_ddp.py expects a *single-cell* AnnData with GEARS-specific columns - it does its own
# internal preprocessing (DE-gene ranking, per-cell graph construction) once those are present:
#   obs['condition']: "GENE+ctrl" for a single perturbation, or exactly "ctrl" for controls
#                     (see gears/utils.py:filter_pert_in_go for the exact format it parses)
#   obs['cell_type']: this file is one cell line (K562), but GEARS still needs the column

#   var['gene_name']: already present in this file under that exact name
# This file's own obs['gene'] holds the perturbed gene per cell, with "non-targeting" for controls.
data_name = 'k562_essential'

print(f"Loading {local_path}...")
adata = ad.read_h5ad(local_path)  # no backed='r' - fully loads into memory

# Once the raw file is fully loaded into memory, its on-disk copy isn't needed for the write
# below, and the reshaped/log1p'd output can end up similarly sized (or larger) than the raw
# input - free it now rather than needing both on disk at once (it's only ~11s to re-download
# from GCS if ever needed again).
os.remove(local_path)
print(f"Freed disk space: removed raw file at {local_path}")

adata.obs['condition'] = adata.obs['gene'].apply(
    lambda g: 'ctrl' if g == 'non-targeting' else f'{g}+ctrl'
)
adata.obs['cell_type'] = 'K562'
assert 'gene_name' in adata.var.columns

# scFoundation's pretrained embeddings are indexed by its own fixed 19,264-gene vocabulary, not
# whatever genes a given dataset happens to measure. PertAdapt's own preprocess/preprocess.py
# does this same remap - without it, MAEAutobinencoder's internal truncation to its 19264-length
# vocabulary (modules/encoders.py) never actually truncates anything for a smaller gene panel
# like this one (8563 genes), leaving 2 extra "resolution append" tokens unstripped and breaking
# a downstream reshape that assumes exactly num_genes positions (confirmed via a standalone
# reproduction before ever running this on real data).
import pandas as pd
import numpy as np
import scipy.sparse

gene_index_path = f"{PERTADAPT_ROOT}/preprocess/OS_scRNA_gene_index.19264.tsv"
canonical_genes = pd.read_csv(gene_index_path, sep='\t')['gene_name'].tolist()
canonical_gene_to_idx = {g: i for i, g in enumerate(canonical_genes)}

overlap_genes, overlap_col_idx = [], []
for j, g in enumerate(adata.var['gene_name'].tolist()):
    if g in canonical_gene_to_idx:
        overlap_genes.append(g)
        overlap_col_idx.append(j)
print(f"Genes overlapping scFoundation's vocabulary: {len(overlap_genes)} / {adata.shape[1]}")

adata = adata[:, overlap_col_idx].copy()
X = scipy.sparse.csr_matrix(adata.X)
old_col_to_new_col = np.array([canonical_gene_to_idx[g] for g in overlap_genes], dtype=np.int32)
remapped_indices = old_col_to_new_col[X.indices]
X_remapped = scipy.sparse.csr_matrix(
    (X.data, remapped_indices, X.indptr), shape=(X.shape[0], len(canonical_genes))
)
X_remapped.sort_indices()

adata = ad.AnnData(X=X_remapped, obs=adata.obs)
adata.var = pd.DataFrame(index=canonical_genes, data={'gene_name': canonical_genes})

# Matches PertAdapt's own reference pipeline: a light cell-quality filter after remapping
# (min_cells=0 for genes, i.e. no gene filtering, is intentionally omitted as a no-op).
sc.pp.filter_cells(adata, min_genes=200)

# Recomputed from the remapped (19264-gene) matrix, matching preprocess/preprocess.py exactly -
# more correct than reusing the raw file's own UMI_count column, which was counted over a
# different (pre-remap) gene set.
total_count = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs['total_count'] = np.round(total_count).astype(np.float32)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# This anndata version gates writing pandas' newer nullable StringArray dtype (used here for
# obs['cell_barcode']/obs index) behind an opt-in setting for backwards compatibility - opt in.
import anndata
anndata.settings.allow_write_nullable_strings = True

data_h5ad_path = f"{PERTADAPT_ROOT}/data/{data_name}.h5ad"
os.makedirs(os.path.dirname(data_h5ad_path), exist_ok=True)
adata.write_h5ad(data_h5ad_path)

print(f"Saved GEARS-formatted AnnData to {data_h5ad_path}")
print(adata.obs['condition'].value_counts().head())

Loading /root/vcc2026-l3ba/data/K562_essential_raw_singlecell_01.h5ad...
Freed disk space: removed raw file at /root/vcc2026-l3ba/data/K562_essential_raw_singlecell_01.h5ad
Saved GEARS-formatted AnnData to /root/vcc2026-l3ba/PertAdapt/data/k562_essential.h5ad
condition
ctrl            10691
RPL3+ctrl        1996
NCBP2+ctrl        992
KIF11+ctrl        974
SLC39A9+ctrl      752
Name: count, dtype: int64


### 9. scFoundation checkpoint

In [5]:
model_checkpoint_dir = f"{PERTADAPT_ROOT}/scFoundation/model/models"
model_checkpoint_path = f"{model_checkpoint_dir}/models.ckpt"
os.makedirs(model_checkpoint_dir, exist_ok=True)

if os.path.exists(model_checkpoint_path):
    size_gb = os.path.getsize(model_checkpoint_path) / 1e9
    print(f"Checkpoint already present: {model_checkpoint_path} ({size_gb:.2f} GB)")
else:
    
    # PertAdapt's own download.txt only lists OneDrive/SharePoint folders, which aren't
    # script-downloadable - this HuggingFace mirror is a direct URL instead.
    model_download_url = 'https://huggingface.co/zhaoxuan-li/scFoundation-model/resolve/main/models/models.ckpt'
    print(f"Downloading from {model_download_url}")
    !wget -O {model_checkpoint_path} {model_download_url}

Checkpoint already present: /root/vcc2026-l3ba/PertAdapt/scFoundation/model/models/models.ckpt (1.43 GB)


### 10. Train

Adapted from `run_norman.sh` for our K562 single-cell data on a single-GPU box:
- `data_dir` / `data_name`: point at the `{data_name}.h5ad` produced above. On first run,
  `train_ddp.py` reads that raw file and runs GEARS' own preprocessing on it (DE-gene ranking,
  one graph per cell - this can take a while for 300k+ cells), caching the result under
  `data/{data_name}/` for subsequent runs.
- `singlecell_model_path`: the checkpoint located in the previous cell.
- `nproc_per_node=1`: a rented GPU box typically has a single GPU.
- `result_dir`: a fresh timestamped directory under `scFoundation/PertAdapter/results/`.

In [20]:
import os
os.environ['ANNDATA_ALLOW_WRITE_NULLABLE_STRINGS'] = '1'


In [23]:
import datetime
import sys

os.chdir(f"{PERTADAPT_ROOT}/scFoundation/PertAdapter/run_sh")

pertadapt_data_dir = f"{PERTADAPT_ROOT}/data/"
os.makedirs(pertadapt_data_dir, exist_ok=True)

gene2go_pkl_path = os.path.join(pertadapt_data_dir, 'gene2go.pkl')
gene2go_full_path = os.path.join(pertadapt_data_dir, 'gene2go_full.pkl')
if not os.path.exists(gene2go_full_path) and not os.path.exists(gene2go_pkl_path):
    gene2go_pkl_url = 'https://raw.githubusercontent.com/biomap-research/scFoundation/main/GEARS/data/gene2go.pkl'
    print(f"Downloading gene2go.pkl -> {gene2go_full_path}")
    !wget -O {gene2go_full_path} {gene2go_pkl_url}
elif not os.path.exists(gene2go_full_path):
    # First time seeing an already-downloaded (unfiltered) file - keep a permanent copy of the
    # full reference before we ever filter it, so re-running this cell later (or switching
    # data_name) can always re-derive a correctly-filtered version instead of compounding filters.
    import shutil
    shutil.copy(gene2go_pkl_path, gene2go_full_path)
print(f"gene2go_full.pkl present: {gene2go_full_path}")

# GEARS' own model_initialize() builds a GO-based gene similarity graph via an O(n^2) nested
# Python loop (gears/utils.py:get_go_auto) over EVERY gene in gene2go.pkl - unfiltered, that's
# the full generic reference (67,832 genes => ~4.6 billion pairwise comparisons => tens of hours,
# confirmed via a standalone reproduction and matching an independent report of ~55h on an A100
# for this same codebase). That gene list is pert_list = pert_data.pert_names, which PertData
# derives directly from gene2go.pkl's own keys - so filtering the *data file* down to just the
# genes this dataset's perturbations actually target shrinks pert_list correspondingly, with zero
# changes to PertAdapt's own code.
import pickle
data_h5ad_path = f"{PERTADAPT_ROOT}/data/{data_name}.h5ad"
adata_obs = ad.read_h5ad(data_h5ad_path, backed='r')
target_genes = {c.split('+')[0] for c in adata_obs.obs['condition'].unique() if c != 'ctrl'}
adata_obs.file.close()

with open(gene2go_full_path, 'rb') as f:
    gene2go_full = pickle.load(f)
gene2go_filtered = {g: v for g, v in gene2go_full.items() if g in target_genes}
print(f"Filtering gene2go.pkl for GO-graph construction: {len(gene2go_full)} -> {len(gene2go_filtered)} genes")
with open(gene2go_pkl_path, 'wb') as f:
    pickle.dump(gene2go_filtered, f)

# data_name was set by the preprocessing cell above; data_dir is relative to run_sh/ here.
data_dir = '../../../data/'
seed = 1
epochs = 1
valid_every = 1
batch_size = 4
accumulation_steps = 2
test_batch_size = 4
hidden_size = 512
bin_set = 'autobin_resolution_append'
model_type = 'maeautobin'
finetune_method = 'frozen'
singlecell_model_path = model_checkpoint_path   # from the checkpoint cell above
mode = 'v1'
highres = 0
lr = 0.01

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
result_base_dir = f"{PERTADAPT_ROOT}/scFoundation/PertAdapter/results/"
result_dir = os.path.join(result_base_dir, f'pa_adj_weighted_de_loss_k562-{timestamp}')
os.makedirs(result_dir, exist_ok=True)
print(f"Result directory: {result_dir}")

command = f"""
{sys.executable} -m torch.distributed.run \
    --standalone \
    --nnodes=1 --nproc_per_node 1 \
    ../train_ddp.py \
    --data_dir={data_dir} \
    --data_name={data_name} \
    --seed={seed} \
    --result_dir={result_dir} \
    --epochs={epochs} \
    --valid_every={valid_every} \
    --batch_size={batch_size} \
    --test_batch_size={test_batch_size} \
    --hidden_size={hidden_size} \
    --bin_set={bin_set} \
    --model_type={model_type} \
    --finetune_method={finetune_method} \
    --singlecell_model_path={singlecell_model_path} \
    --mode={mode} \
    --highres={highres} \
    --accumulation_steps={accumulation_steps} \
    --loss='loss_adapt' \
    --ddp_loss_weight=5.0 \
    --lr={lr} \
    --model_class="GEARS_Model_Pert_Adapter_New" \
    --proj_name="PertAdapt" \
    --exp_name="pa_adj_weighted_de_loss_k562-{timestamp}"
"""
print(command)

import subprocess
log_path = os.path.join(result_dir, 'train.log')
with open(log_path, 'wb') as logf:
    proc = subprocess.Popen(
        command, shell=True, stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,  # detach from the Jupyter kernel's session so training survives
                                  # a VS Code/SSH disconnect instead of being SIGKILLed with it
    )
print(f"Launched training as a detached background process, PID={proc.pid}.")
print("This keeps running even if VS Code disconnects or the kernel restarts.")
print(f"Monitor with: tail -f {log_path}")
print(f"result_dir = {result_dir}")

gene2go.pkl already present: /root/vcc2026-l3ba/PertAdapt/data/gene2go.pkl
Result directory: /root/vcc2026-l3ba/PertAdapt/scFoundation/PertAdapter/results/pa_adj_weighted_de_loss_k562-2026-09-30_06-14-10

/usr/bin/python3.13 -m torch.distributed.run     --standalone     --nnodes=1 --nproc_per_node 1     ../train_ddp.py     --data_dir=../../../data/     --data_name=k562_essential     --seed=1     --result_dir=/root/vcc2026-l3ba/PertAdapt/scFoundation/PertAdapter/results/pa_adj_weighted_de_loss_k562-2026-09-30_06-14-10     --epochs=1     --valid_every=1     --batch_size=8     --test_batch_size=4     --hidden_size=512     --bin_set=autobin_resolution_append     --model_type=maeautobin     --finetune_method=frozen     --singlecell_model_path=/root/vcc2026-l3ba/PertAdapt/scFoundation/model/models/models.ckpt     --mode=v1     --highres=0     --accumulation_steps=2     --loss='loss_adapt'     --ddp_loss_weight=5.0     --lr=0.01     --model_class="GEARS_Model_Pert_Adapter_New"     --proj_name

### 11. Check the training log

In [24]:
log_file_path = os.path.join(result_dir, 'train.log')

if os.path.exists(log_file_path):
    print(f"Contents of {log_file_path}:")
    with open(log_file_path, 'r') as f:
        print(f.read())
else:
    print(f"Error: Log file not found at {log_file_path}")

Contents of /root/vcc2026-l3ba/PertAdapt/scFoundation/PertAdapter/results/pa_adj_weighted_de_loss_k562-2026-09-30_06-14-10/train.log:
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DDP init!
local_rank:  0
read ../../../data/gene2go.pkl
load data
These perturbations are not in the GO graph and is thus not able to make prediction for...
['C7orf26+ctrl', 'C19orf53+ctrl', 'KRTAP4-7+ctrl', 'OR4F4+ctrl', 'C16orf86+ctrl', ..., 'FAM229A+ctrl', 'SEM1+ctrl', 'GNB1L+ctrl', 'MTRNR2L1+ctrl', 'FAU+ctrl']
Length: 15
Categories (2058, string): ['AAAS+ctrl', 'AAMP+ctrl', 'AARS+ctrl', 'AARS2+ctrl', ..., 'ZRSR2+ctrl',
                            'ZW10+ctrl', 'ZWINT+ctrl', 'ctrl']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:0
combo_seen1:0
combo_seen2:0
unseen_single:

### Next steps

Training writes checkpoints and metrics under `result_dir`. Evaluating or running inference with
the trained adapter isn't wired up in this notebook yet - see PertAdapt's `results_process/`
directory and its README for the expected evaluation workflow.